# Background knowledge

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/knowledge.ipynb)

You often know part of the answer before you look at the data: a cause comes before its effect in
time, a treatment was given before the outcome was measured, or an earlier study settled one
link. `andrey.Knowledge` passes what you know to a search, which then returns only graphs that
agree with it. Here PC finds every pair of a simulated graph but cannot orient a single edge;
ordering the variables in time orients nearly all of them.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.11.0` from PyPI;
local kernels use the installed environment.

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[viz]==0.11.0" pandas

In [2]:
import pandas as pd

import andrey
from andrey.data import sample_scm
from andrey.metrics import shd

A linear-Gaussian model with 10 variables, whose true graph we know. `sample_scm` lists the
columns in an order drawn from the seed, so their order says nothing about the causal order.

In [3]:
sim = sample_scm(d=10, n=2000, density=2, seed=5)
names = [f"x{i}" for i in range(10)]
data = pd.DataFrame(sim.data, columns=names)
truth = sim.graph
print("true edges:", ", ".join(f"{names[a]} -> {names[b]}" for a, b, _ in truth.oriented_edges()))

true edges: x1 -> x0, x0 -> x8, x9 -> x0, x9 -> x1, x7 -> x2, x5 -> x3, x4 -> x6, x7 -> x4, x8 -> x4, x8 -> x7


## Without knowledge

PC finds all ten pairs, but the data alone cannot say which way any of them points: every edge
is undirected, and each counts as an error against the true graph.

In [4]:
out = andrey.pc(data)
print(out)
print("errors:", shd(out.structure, truth))

pc  cpdag  |  10 nodes  |  10 edges (10 undirected)

  x0 -- x1
  x0 -- x8
  x0 -- x9
  x1 -- x9
  x2 -- x7
  x3 -- x5
  x4 -- x6
  x4 -- x7
  x4 -- x8
  x7 -- x8

  indep_test: fisherz
errors: 10


## Tiers

Tiers group the variables in time: no edge may point from a later tier into an earlier one. Say
the first three variables were measured at the start, the next four in the middle, and the last
three at the end. In this simulation the tiers follow the true causal order, as a domain
expert's would.

In [5]:
knowledge = andrey.Knowledge(
    tiers=[["x9", "x5", "x1"], ["x3", "x0", "x8", "x7"], ["x2", "x4", "x6"]]
)
guided = andrey.pc(data, knowledge=knowledge)
print(guided)
print("errors:", shd(guided.structure, truth))

pc  cpdag  |  10 nodes  |  10 edges (9 directed, 1 undirected)

  x1 -> x0
  x0 -> x8
  x9 -> x0
  x1 -- x9
  x7 -> x2
  x5 -> x3
  x4 -> x6
  x7 -> x4
  x8 -> x4
  x8 -> x7

  indep_test: fisherz
errors: 1


In [6]:
import andrey.viz

andrey.viz.draw(guided.structure)

'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 664 240" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="664" height="240" fill="#fafbfc"/><line x1="209.00" y1="151.64" x2="45.00" y2="152.96" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,151.56 208.96,146.64 209.04,156.64" fill="#3a4658" stroke-linejoin="round"/><line x1="245.00" y1="151.45" x2="309.00" y2="151.45" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.00,151.45 309.00,146.45 309.00,156.45" fill="#3a4658" stroke-linejoin="round"/><line x1="210.11" y1="158.51" x2="144.37" y2="179.72" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.63,155.44 208.58,153.75 211.65,163.27" fill="#3a4658" stroke-linejoin="round"/><line x1="44.43" y1="156.87" x2="119.57" y2="179.90" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="509.03" y1="187.44" x2="444.98" y2="184.34" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="519.02,187.92 509.27,182.44 508.79,192.43" fill="#3a4658" stroke-linejoin="round"/><line x1="109.00" y1="51.45" x2="45.00" y2="51.45" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.00,51.45 109.00,46.45 109.00,56.45" fill="#3a4658" stroke-linejoin="round"/><line x1="545.00" y1="117.58" x2="609.00" y2="117.58" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="619.00,117.58 609.00,112.58 609.00,122.58" fill="#3a4658" stroke-linejoin="round"/><line x1="512.82" y1="130.27" x2="442.84" y2="176.54" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="521.16,124.75 510.06,126.10 515.57,134.44" fill="#3a4658" stroke-linejoin="round"/><line x1="509.32" y1="121.42" x2="344.82" y2="149.28" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="519.18,119.75 508.49,116.49 510.16,126.35" fill="#3a4658" stroke-linejoin="round"/><line x1="410.11" y1="176.65" x2="344.37" y2="155.44" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.63,179.72 411.65,171.89 408.58,181.41" fill="#3a4658" stroke-linejoin="round"/><circle cx="232.00" cy="151.45" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="151.45" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x0</text><circle cx="32.00" cy="153.06" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="153.06" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x1</text><circle cx="532.00" cy="188.55" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="532.00" y="188.55" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x2</text><circle cx="132.00" cy="51.45" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="51.45" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x3</text><circle cx="532.00" cy="117.58" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="532.00" y="117.58" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x4</text><circle cx="32.00" cy="51.45" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="51.45" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x5</text><circle cx="632.00" cy="117.58" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="632.00" y="117.58" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">x6</text><circle cx="432.00" cy="183.71" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="432.00" y="183.71" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="cent

## Required and forbidden edges

When you know one edge, require it; when you know an edge cannot exist, forbid it. Forbidding
`a -> b` forbids that direction only, so forbid both to keep two variables apart. PC then orients
the edges these imply.

In [7]:
knowledge = andrey.Knowledge(required=[("x8", "x7")], forbidden=[("x6", "x4")])
out = andrey.pc(data, knowledge=knowledge)
print(out)
print("errors:", shd(out.structure, truth))

pc  cpdag  |  10 nodes  |  10 edges (7 undirected, 3 directed)

  x0 -- x1
  x0 -- x8
  x0 -- x9
  x1 -- x9
  x7 -> x2
  x3 -- x5
  x4 -> x6
  x4 -- x7
  x4 -- x8
  x8 -> x7

  indep_test: fisherz
errors: 7


## Every family

The same `Knowledge` works across families: the constraint-based, score-based, and
permutation-based searches, the LiNGAM methods, and the time-series methods, where a link can
also name its lag. Each method's reference page says which parts it honors.

In [8]:
knowledge = andrey.Knowledge(
    tiers=[["x9", "x5", "x1"], ["x3", "x0", "x8", "x7"], ["x2", "x4", "x6"]]
)
for method, settings in [("pc", {}), ("ges", {}), ("boss", {"seed": 0}), ("grasp", {"seed": 0})]:
    fit = getattr(andrey, method)
    before = shd(fit(data, **settings).structure, truth)
    after = shd(fit(data, knowledge=knowledge, **settings).structure, truth)
    print(f"{method:>5}: {before} errors without knowledge, {after} with tiers")

   pc: 10 errors without knowledge, 1 with tiers
  ges: 10 errors without knowledge, 1 with tiers
 boss: 10 errors without knowledge, 1 with tiers
grasp: 10 errors without knowledge, 1 with tiers


## Wrong knowledge

A search treats knowledge as fact. Require an edge that is not in the true graph, and the result
has it:

In [9]:
wrong = andrey.pc(data, knowledge=andrey.Knowledge(required=[("x9", "x3")]))
print("errors:", shd(wrong.structure, truth))

errors: 11


So pass only what you are sure of. [Background knowledge](../guides/getting-started.md#background-knowledge)
in the getting-started guide covers every part of `Knowledge`.